# Подготовка окружения

In [ ]:
# %pip install ultralytics -q

from pathlib import Path
import json
import shutil
from collections import OrderedDict

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown

from ultralytics.models.yolo.model import YOLO

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 50)

print('Окружение подготовлено.')

Окружение подготовлено.


# Проверка путей

In [22]:
def detect_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'runs').exists() and (candidate / 'data').exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = detect_project_root()
MODEL_PATH = PROJECT_ROOT / 'runs' / 'train' / 'yolo11n_20260518_170929' / 'weights' / 'best.pt'
DATA_YAML_CANDIDATES = [
    PROJECT_ROOT / 'dataset' / 'data.yaml',
    PROJECT_ROOT / 'data' / 'datasets' / 'tomatoes_v1' / 'data.yaml',
]
TEST_IMAGES_CANDIDATES = [
    PROJECT_ROOT / 'dataset' / 'test' / 'images',
    PROJECT_ROOT / 'data' / 'datasets' / 'tomatoes_v1' / 'test' / 'images',
]
RESULTS_DIR = PROJECT_ROOT / 'models'
EVAL_OUTPUT_DIR = PROJECT_ROOT / 'evaluation_output'
PREDICTIONS_DIR = EVAL_OUTPUT_DIR / 'predictions'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML_PATH = next((p for p in DATA_YAML_CANDIDATES if p.exists()), None)
TEST_IMAGES_DIR = next((p for p in TEST_IMAGES_CANDIDATES if p.exists()), None)

print('PROJECT_ROOT:', PROJECT_ROOT.resolve())
print('MODEL_PATH:', MODEL_PATH.resolve())
print('DATA_YAML_PATH:', DATA_YAML_PATH.resolve() if DATA_YAML_PATH else 'не найден')
print('TEST_IMAGES_DIR:', TEST_IMAGES_DIR.resolve() if TEST_IMAGES_DIR else 'не найден')
print('RESULTS_DIR:', RESULTS_DIR.resolve())
print('PREDICTIONS_DIR:', PREDICTIONS_DIR.resolve())

checks = OrderedDict([
    ('best.pt', MODEL_PATH.exists()),
    ('data.yaml', DATA_YAML_PATH is not None),
    ('test images dir', TEST_IMAGES_DIR is not None),
])

display(pd.DataFrame([{'Путь': k, 'Найден': 'Да' if v else 'Нет'} for k, v in checks.items()]))

PROJECT_ROOT: C:\Users\User\Desktop\vegvision-yolo
MODEL_PATH: C:\Users\User\Desktop\vegvision-yolo\runs\train\yolo11n_20260518_170929\weights\best.pt
DATA_YAML_PATH: C:\Users\User\Desktop\vegvision-yolo\data\datasets\tomatoes_v1\data.yaml
TEST_IMAGES_DIR: C:\Users\User\Desktop\vegvision-yolo\data\datasets\tomatoes_v1\test\images
RESULTS_DIR: C:\Users\User\Desktop\vegvision-yolo\models
PREDICTIONS_DIR: C:\Users\User\Desktop\vegvision-yolo\evaluation_output\predictions


,Путь,Найден
0,best.pt,Да
1,data.yaml,Да
2,test images dir,Да


# Загрузка модели

In [23]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Не найден файл весов: {MODEL_PATH}')

model = YOLO(str(MODEL_PATH))
print('Модель успешно загружена:', MODEL_PATH.name)

if DATA_YAML_PATH is None:
    raise FileNotFoundError('Не найден файл data.yaml в ожидаемых местах.')

print('Датасет для валидации:', DATA_YAML_PATH.resolve())

Модель успешно загружена: best.pt
Датасет для валидации: C:\Users\User\Desktop\vegvision-yolo\data\datasets\tomatoes_v1\data.yaml


# Валидация модели

In [30]:
if DATA_YAML_PATH is None:
    raise FileNotFoundError('data.yaml не найден, валидация невозможна.')

print('Использую YAML для валидации:', DATA_YAML_PATH.resolve())

results = model.val(
    data=str(DATA_YAML_PATH),
    split='test',
    plots=True,
    save_json=False,
    verbose=False,
)

print('Валидация завершена.')

Использую YAML для валидации: C:\Users\User\Desktop\vegvision-yolo\data\datasets\tomatoes_v1\data.yaml
Ultralytics 8.4.34  Python-3.12.10 torch-2.11.0+cpu CPU (AMD Athlon Silver 3050U with Radeon Graphics)
val: Fast image access  (ping: 0.10.0 ms, read: 56.729.0 MB/s, size: 47.2 KB)
val: Fast image access  (ping: 0.10.0 ms, read: 56.729.0 MB/s, size: 47.2 KB)
val: Scanning C:\Users\User\Desktop\vegvision-yolo\data\datasets\tomatoes_v1\test\labels.cache... 317 images, 0 backgrounds, 278 corrupt: 100% ━━━━━━━━━━━━ 317/317  0.0s
val: Scanning C:\Users\User\Desktop\vegvision-yolo\data\datasets\tomatoes_v1\test\labels.cache... 317 images, 0 backgrounds, 278 corrupt: 100% ━━━━━━━━━━━━ 317/317  0.0s
val: C:\Users\User\Desktop\vegvision-yolo\data\datasets\tomatoes_v1\test\images\151b36defa8ccfe2fd90f4a342affd5a-tobacco-mosaic-virus-the-two_jpg.rf.0b05eff0817196b125b0873831f87d87.jpg: ignoring corrupt image/label: Label class 5 exceeds dataset class count 1. Possible class labels are 0-0
val: C

# Основные метрики

In [25]:
def safe_get(obj, path, default=None):
    current = obj
    for key in path:
        if current is None:
            return default
        if isinstance(current, dict):
            current = current.get(key, default)
        else:
            current = getattr(current, key, default)
    return default if current is None else current

def to_float(value, default=None):
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default

results_dict = getattr(results, 'results_dict', {}) if 'results' in locals() else {}
precision = to_float(safe_get(results, ['box', 'mp']), to_float(results_dict.get('metrics/precision(B)')) if isinstance(results_dict, dict) else None)
recall = to_float(safe_get(results, ['box', 'mr']), to_float(results_dict.get('metrics/recall(B)')) if isinstance(results_dict, dict) else None)
map50 = to_float(safe_get(results, ['box', 'map50']), to_float(results_dict.get('metrics/mAP50(B)')) if isinstance(results_dict, dict) else None)
map5095 = to_float(safe_get(results, ['box', 'map']), to_float(results_dict.get('metrics/mAP50-95(B)')) if isinstance(results_dict, dict) else None)

metrics_df = pd.DataFrame([
    {'Метрика': 'Precision', 'Значение': precision},
    {'Метрика': 'Recall', 'Значение': recall},
    {'Метрика': 'mAP50', 'Значение': map50},
    {'Метрика': 'mAP50-95', 'Значение': map5095},
])
display(metrics_df)

metrics_payload = {
    'model_path': str(MODEL_PATH.resolve()),
    'data_yaml_path': str(DATA_YAML_PATH.resolve()) if DATA_YAML_PATH else None,
    'precision': precision,
    'recall': recall,
    'map50': map50,
    'map50_95': map5095,
    'results_dict': results_dict,
    'save_dir': str(getattr(results, 'save_dir', '')) if 'results' in locals() else None,
}

txt_path = RESULTS_DIR / 'evaluation_results.txt'
json_path = RESULTS_DIR / 'evaluation_results.json'

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write('Оценка модели YOLO11n\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'mAP50: {map50}\n')
    f.write(f'mAP50-95: {map5095}\n')
    f.write(f'Data YAML: {DATA_YAML_PATH.resolve() if DATA_YAML_PATH else "не найден"}\n')
    f.write(f'Weights: {MODEL_PATH.resolve()}\n')

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, ensure_ascii=False, indent=2, default=str)

print('Сохранены файлы:')
print(txt_path.resolve())
print(json_path.resolve())

,Метрика,Значение
0,Precision,0.881210
1,Recall,0.741862
2,mAP50,0.870471
3,mAP50-95,0.695491


Сохранены файлы:
C:\Users\User\Desktop\vegvision-yolo\models\evaluation_results.txt
C:\Users\User\Desktop\vegvision-yolo\models\evaluation_results.json


# Графики качества

In [ ]:
def plot_confusion_matrix_manual(results):
    """Нарисовать confusion matrix на основе results."""
    if not hasattr(results, 'confusion_matrix') or results.confusion_matrix is None:
        print('Confusion matrix недоступна')
        return
    
    cm = results.confusion_matrix.matrix
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, cmap='Blues', aspect='auto')
    plt.colorbar(label='Количество образцов')
    plt.title('Confusion Matrix')
    plt.xlabel('Предсказанный класс')
    plt.ylabel('Истинный класс')
    plt.tight_layout()
    plt.show()

def plot_metrics_curves(results):
    """Нарисовать кривые precision, recall, F1."""
    if not hasattr(results, 'curves') or results.curves is None:
        print('Кривые недоступны')
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    print('Доступные атрибуты results:', [attr for attr in dir(results) if not attr.startswith('_')])

if 'results' in locals() and results is not None:
    print('Попытка отрисовки графиков...')
    plot_confusion_matrix_manual(results)
    plot_metrics_curves(results)
else:
    print('Результаты валидации недоступны')

Попытка отрисовки графиков...


<Figure size 1000x800 with 2 Axes>

Доступные атрибуты results: ['ap_class_index', 'box', 'class_result', 'clear_stats', 'confusion_matrix', 'curves', 'curves_results', 'fitness', 'keys', 'maps', 'mean_results', 'names', 'nt_per_class', 'nt_per_image', 'process', 'results_dict', 'save_dir', 'speed', 'stats', 'summary', 'to_csv', 'to_df', 'to_json', 'update_stats']


# Визуализация предсказаний

In [27]:
if TEST_IMAGES_DIR is None:
    print('Папка с тестовыми изображениями не найдена.')
else:
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    test_images = sorted([p for p in TEST_IMAGES_DIR.iterdir() if p.is_file() and p.suffix.lower() in image_exts])[:5]
    if not test_images:
        print('В test/images нет изображений.')
    else:
        if PREDICTIONS_DIR.exists():
            shutil.rmtree(PREDICTIONS_DIR)
        PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

        print('Запускаю предсказания для изображений:')
        for img_path in test_images:
            print(' -', img_path.name)

        pred_results = model.predict(
            source=[str(p) for p in test_images],
            save=True,
            project=str(EVAL_OUTPUT_DIR),
            name='predictions',
            exist_ok=True,
            verbose=False,
        )

        predicted_files = sorted([p for p in PREDICTIONS_DIR.iterdir() if p.is_file()])
        print('Сохранено предсказаний:', len(predicted_files))
        for path in predicted_files[:5]:
            show_image(path, title=path.name, figsize=(10, 8))

        if not predicted_files:
            print('Файлы предсказаний не найдены.')

Запускаю предсказания для изображений:
 - 039b47d574bc4bb8a14259a1cd96a741_jpg.rf.b8b089bec8cfbe62eed8409947b888bc.jpg
 - 151b36defa8ccfe2fd90f4a342affd5a-tobacco-mosaic-virus-the-two_jpg.rf.0b05eff0817196b125b0873831f87d87.jpg
 - 1c6727083428411181e9b58b4778c6b7_jpg.rf.6935942a2138dd0af594582199644691.jpg
 - 1cee26b4-0ba1-4371-8c7e-c641e6ca311d___PSU_CG-2283_JPG.rf.7c7d77b6d1d953aca1cc9c7958493f26.jpg
 - 1d89d788-438c-4cee-9403-6fd8ac9863de___PSU_CG-2254_JPG.rf.88bef7f49cc5aa0d8d0bf89da4a90719.jpg
Results saved to C:\Users\User\Desktop\vegvision-yolo\evaluation_output\predictions
Results saved to C:\Users\User\Desktop\vegvision-yolo\evaluation_output\predictions
Сохранено предсказаний: 5
Сохранено предсказаний: 5


<Figure size 1500x400 with 3 Axes>

<Figure size 1000x800 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

# Сохранение результатов

In [28]:
summary_df = metrics_df.copy()
summary_df['Значение'] = summary_df['Значение'].map(lambda x: None if x is None else round(float(x), 6))
summary_df['Описание'] = [
    'Средняя точность детектирования',
    'Средняя полнота детектирования',
    'mAP при IoU=0.50',
    'mAP при IoU=0.50:0.95',
]
display(summary_df)

summary_csv_path = RESULTS_DIR / 'evaluation_results_table.csv'
summary_df.to_csv(summary_csv_path, index=False, encoding='utf-8')
print('Сохранена таблица метрик:', summary_csv_path.resolve())
print('Папка с предсказаниями:', PREDICTIONS_DIR.resolve())

,Метрика,Значение,Описание
0,Precision,0.881210,Средняя точность детектирования
1,Recall,0.741862,Средняя полнота детектирования
2,mAP50,0.870471,mAP при IoU=0.50
3,mAP50-95,0.695491,mAP при IoU=0.50:0.95


Сохранена таблица метрик: C:\Users\User\Desktop\vegvision-yolo\models\evaluation_results_table.csv
Папка с предсказаниями: C:\Users\User\Desktop\vegvision-yolo\evaluation_output\predictions


# Итоговый вывод

In [29]:
def metric_text(value):
    return 'недоступно' if value is None else f'{value:.4f}'

conclusion = f'''
**Краткий вывод по модели YOLO11n**

- Precision: {metric_text(precision)}
- Recall: {metric_text(recall)}
- mAP50: {metric_text(map50)}
- mAP50-95: {metric_text(map5095)}

Модель успешно загружена, прошла валидацию и сформировала предсказания для первых тестовых изображений.
Все основные результаты сохранены в папке `models`, а предсказания — в `evaluation_output/predictions`.
'''

display(Markdown(conclusion))


**Краткий вывод по модели YOLO11n**

- Precision: 0.8812
- Recall: 0.7419
- mAP50: 0.8705
- mAP50-95: 0.6955

Модель успешно загружена, прошла валидацию и сформировала предсказания для первых тестовых изображений.
Все основные результаты сохранены в папке `models`, а предсказания — в `evaluation_output/predictions`.
